In [52]:
# ---------------------------
# PATH SETUP
# ---------------------------
import os
import pandas as pd
import numpy as np

BASE_PATH = "../"
DATA_PATH = os.path.join(BASE_PATH, "data/processed/")
OUTPUT_PATH = os.path.join(BASE_PATH, "outputs/")

# Create output folders if not exist
os.makedirs(os.path.join(OUTPUT_PATH, "predictions"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_PATH, "feature_importance"), exist_ok=True)

print("Paths ready")

Paths ready


Import the necessary packages

In [53]:
from sklearn.model_selection import LeaveOneOut
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

# Optional XGBoost
try:
    from xgboost import XGBRegressor
    use_xgb = True
except:
    print("XGBoost not installed → skipping")
    use_xgb = False

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

Load the correct data

In [54]:
df_match = pd.read_csv(DATA_PATH + "clean_match.csv")
df_context = pd.read_csv(DATA_PATH + "clean_context.csv")
df_tickets = pd.read_csv(DATA_PATH + "clean_tickets.csv")
df_trends = pd.read_csv(DATA_PATH + "clean_trends.csv")
df_articles = pd.read_csv(DATA_PATH + "clean_articles.csv")

# Convert dates
df_match['match_date'] = pd.to_datetime(df_match['match_date'])
df_context['match_date'] = pd.to_datetime(df_context['match_date'])
df_trends['date'] = pd.to_datetime(df_trends['date'])
df_articles['date'] = pd.to_datetime(df_articles['date'])

print("Data loaded")

Data loaded


Merging Data

In [55]:
# Keep only home matches
df = df_match[df_match['is_home_match'] == True].copy()

# Merge
df = df.merge(df_tickets, on='match_id', how='left')
df = df.merge(df_context, on='match_id', how='left')

# Fix duplicate match_date columns
df['match_date'] = df['match_date_x']  # use match_date from df_match

# Optional: drop duplicates
df = df.drop(columns=['match_date_x', 'match_date_y'], errors='ignore')

# Trends
df_trends_agg = df_trends.groupby('match_id')['ohl_interest'].mean().reset_index()
df = df.merge(df_trends_agg, on='match_id', how='left')

# Articles
df_articles_agg = df_articles.groupby('match_id').size().reset_index(name='article_count')
df = df.merge(df_articles_agg, on='match_id', how='left')

df['article_count'] = df['article_count'].fillna(0)

# Sort
df = df.sort_values('match_date').reset_index(drop=True)

print("Merge complete:", df.shape)

Merge complete: (71, 71)


Feature Engineering

This is for the Sporting

In [56]:
points_map = {'W': 3, 'D': 1, 'L': 0}
df['points'] = df['result_home'].map(points_map)

df['points_last_5'] = df['points'].rolling(5).sum().shift(1)

df['win'] = (df['result_home'] == 'W').astype(int)
df['wins_last_3'] = df['win'].rolling(3).sum().shift(1)

df['goal_diff'] = df['goals_home_ft'] - df['goals_away_ft']
df['goal_diff_last_5'] = df['goal_diff'].rolling(5).sum().shift(1)

Let us also use Timing

In [57]:
df['matchday'] = pd.to_numeric(df['matchday'], errors='coerce')
df['season_progress'] = df['matchday'] / df['matchday'].max()

How important the match is

In [58]:
# Approximate strength
df['form_strength'] = df['points_last_5']

# Normalize
df['form_strength_norm'] = (
    (df['form_strength'] - df['form_strength'].min()) /
    (df['form_strength'].max() - df['form_strength'].min())
)

# Combine with season pressure
df['match_importance'] = (
    0.5 * df['form_strength_norm'] +
    0.5 * df['season_progress']
)

# Binary version
df['is_high_importance'] = (df['match_importance'] > df['match_importance'].median()).astype(int)

Opponent

In [59]:
df['opponent'] = df['away_team']
df['opponent_freq'] = df['opponent'].map(df['opponent'].value_counts())

Promotions

In [60]:
df['has_promotion'] = df['has_promotion'].astype(int)

Lag

In [61]:
df['attendance_lag_1'] = df['tickets_scanned'].shift(1)

Final Dataset:

In [62]:
features = [
    'points_last_5',
    'wins_last_3',
    'goal_diff_last_5',

    'match_importance',        
    'is_high_importance',

    'is_weekend',
    'season_progress',

    'opponent_freq',

    'has_promotion',

    'attendance_lag_1'
]

target = 'tickets_scanned'

df_model = df[features + [target]].dropna()

X = df_model[features]
y = df_model[target]

print("Final dataset:", df_model.shape)

Final dataset: (66, 11)


Baseline

In [63]:
baseline_pred = np.full(len(y), y.mean())

baseline_results = {
    "MAE": mean_absolute_error(y, baseline_pred),
    "RMSE": np.sqrt(mean_squared_error(y, baseline_pred)),
    "R2": r2_score(y, baseline_pred),
    "MAPE": np.mean(np.abs((y - baseline_pred) / y)) * 100
}

Models + LOOCV

In [66]:
models = {
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
}

if use_xgb:
    models["XGBoost"] = XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.1, random_state=42)

loo = LeaveOneOut()

results = {}
predictions_dict = {}

for name, model in models.items():
    y_true, y_pred = [], []
    
    for train_idx, test_idx in loo.split(X):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        
        y_true.append(y_test.values[0])
        y_pred.append(pred[0])
    
    results[name] = {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
        "MAPE": np.mean(np.abs((np.array(y_true) - np.array(y_pred)) / np.array(y_true))) * 100
    }
    
    predictions_dict[name] = pd.DataFrame({
        "Actual": y_true,
        "Predicted": y_pred
    })

Result:

In [68]:
results["Baseline"] = baseline_results

results_df = pd.DataFrame(results).T
print(results_df)

results_df.to_csv(OUTPUT_PATH + "model_results.csv")

                          MAE         RMSE        R2       MAPE
LinearRegression  1369.300507  1731.515992  0.252905  21.298721
RandomForest      1287.489273  1671.548747  0.303756  20.646473
XGBoost           1449.028894  1945.061091  0.057265  23.226588
Baseline          1682.815427  2003.265210  0.000000  26.371874


Feature Importance:

In [69]:
# Linear
lin_model = LinearRegression().fit(X, y)
lin_importance = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": lin_model.coef_
}).sort_values(by="Coefficient", key=abs, ascending=False)

lin_importance.to_csv(OUTPUT_PATH + "feature_importance/linear.csv", index=False)

# Random Forest
rf_model = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
rf_model.fit(X, y)

rf_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_model.feature_importances_
}).sort_values(by="Importance", ascending=False)

rf_importance.to_csv(OUTPUT_PATH + "feature_importance/random_forest.csv", index=False)

Predictions

In [70]:
for name, preds in predictions_dict.items():
    preds.to_csv(OUTPUT_PATH + f"predictions/predictions_{name}.csv", index=False)

# Improved Model: Match Attractiveness Feature

We observed that the model struggles to predict extreme attendance values, particularly high-demand matches. This suggests that key drivers such as opponent attractiveness are not fully captured.

To address this, we introduce a new feature called "match attractiveness", which combines sporting performance and opponent strength.

## Create a new feature (Opponent)

In [71]:
top_teams = ["Club Brugge", "Anderlecht", "STVV", "KV Mechelen", "Westerlo"]

df['is_top_opponent'] = df['away_team'].isin(top_teams).astype(int)

df['opponent_strength_norm'] = (
    (df['opponent_freq'] - df['opponent_freq'].min()) /
    (df['opponent_freq'].max() - df['opponent_freq'].min())
)

df['match_attractiveness'] = (
    0.4 * df['match_importance'] +
    0.4 * df['opponent_strength_norm'] +
    0.2 * df['is_top_opponent']
)

df['form_x_opponent'] = df['points_last_5'] * df['is_top_opponent']

New set:

In [72]:
features_new = [
    'points_last_5',
    'wins_last_3',
    'goal_diff_last_5',

    'match_importance',
    'match_attractiveness',
    'form_x_opponent',

    'is_weekend',
    'season_progress',
    'has_promotion',
    'attendance_lag_1'
]

df_model_new = df[features_new + [target]].dropna()

X_new = df_model_new[features_new]
y_new = df_model_new[target]

Retrain Models

In [73]:
models = {
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
}

if use_xgb:
    models["XGBoost"] = XGBRegressor(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.1,
        random_state=42
    )

# ---------------------------
# LOOCV
# ---------------------------
loo = LeaveOneOut()

results_new = {}
predictions_new_dict = {}

for name, model in models.items():
    y_true, y_pred = [], []
    
    for train_idx, test_idx in loo.split(X_new):
        X_train, X_test = X_new.iloc[train_idx], X_new.iloc[test_idx]
        y_train, y_test = y_new.iloc[train_idx], y_new.iloc[test_idx]
        
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        
        y_true.append(y_test.values[0])
        y_pred.append(pred[0])
    
    results_new[name] = {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
        "MAPE": np.mean(np.abs((np.array(y_true) - np.array(y_pred)) / np.array(y_true))) * 100
    }
    
    predictions_new_dict[name] = pd.DataFrame({
        "Actual": y_true,
        "Predicted": y_pred
    })

# ---------------------------
# RESULTS TABLE
# ---------------------------
results_new_df = pd.DataFrame(results_new).T

print(results_new_df)

                          MAE         RMSE        R2       MAPE
LinearRegression  1346.303993  1650.798361  0.320935  20.837109
RandomForest      1331.207626  1628.754619  0.338950  21.250655
XGBoost           1436.383108  1765.700697  0.223114  23.144354


Comparing the results:

In [74]:
print("OLD MODEL")
print(results_df)

print("\nNEW MODEL")
print(results_new_df)

OLD MODEL
                          MAE         RMSE        R2       MAPE
LinearRegression  1369.300507  1731.515992  0.252905  21.298721
RandomForest      1287.489273  1671.548747  0.303756  20.646473
XGBoost           1449.028894  1945.061091  0.057265  23.226588
Baseline          1682.815427  2003.265210  0.000000  26.371874

NEW MODEL
                          MAE         RMSE        R2       MAPE
LinearRegression  1346.303993  1650.798361  0.320935  20.837109
RandomForest      1331.207626  1628.754619  0.338950  21.250655
XGBoost           1436.383108  1765.700697  0.223114  23.144354
